# Convert a LIBERO HDF5 task to LeRobot

This notebook inspects a downloaded official task, creates an explicit plan, converts one demonstration, and validates the target. Only the optional second conversion and merge are gated by `RUN_MERGE`.

In [1]:
from pathlib import Path
from pprint import pprint

from leport import convert, create_plan, inspect, merge, validate
from leport.conversion.plan import save_plan
from leport.sources import EpisodeSelection

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("Project root not found; start from the leport root or notebooks/")

source_path = (
    project_root / "data/libero/libero_90/KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet_demo.hdf5"
)
output_root = project_root / "outputs"
plan_path = output_root / "libero-close-drawer-demo.yaml"
target_root = output_root / "libero-close-drawer-demo"

print("Configured source:", source_path)

Configured source: /Users/xzhu/Documents/VLA/leport/data/libero/libero_90/KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet_demo.hdf5


## 1. Inspect the official source

Inspection validates LIBERO task metadata and reports qualified demo IDs, field schemas, camera shapes, and source attributes without loading complete arrays.

In [2]:
# Fail clearly instead of manufacturing notebook data.
if not source_path.is_file():
    raise FileNotFoundError(f"Official LIBERO source not found: {source_path}")
inspection = inspect(source_path, adapter="libero")
# Keep the committed execution result readable while retaining the important source facts.
pprint(
    {
        "adapter": inspection.adapter,
        "episode_count": len(inspection.episode_ids),
        "first_episode_ids": inspection.episode_ids[:5],
        "total_frames": inspection.total_frames,
        "field_count": len(inspection.fields),
        "field_selectors": [field.selector for field in inspection.fields],
        "diagnostics": inspection.diagnostics,
    }
)

{'adapter': 'libero',
 'diagnostics': (),
 'episode_count': 50,
 'field_count': 12,
 'field_selectors': ['actions',
                     'dones',
                     'obs/agentview_rgb',
                     'obs/ee_ori',
                     'obs/ee_pos',
                     'obs/ee_states',
                     'obs/eye_in_hand_rgb',
                     'obs/gripper_states',
                     'obs/joint_states',
                     'rewards',
                     'robot_states',
                     'states'],
 'first_episode_ids': ('KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet/demo_0',
                       'KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet/demo_1',
                       'KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet/demo_2',
                       'KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet/demo_3',
                       'KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet/demo_4'),
 'total_frames': 3762}


## 2. Define explicit conversion semantics

The language instruction comes from source metadata. FPS, robot type, action meaning, casts, selector order, and target feature names remain explicit choices.

In [3]:
task_name = "KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet"
episode_selection = EpisodeSelection(episode_ids=(f"{task_name}/demo_0",))
fps = 20
action_source = "actions"
state_sources = ("obs/ee_states", "obs/gripper_states")
image_sources = {
    "obs/agentview_rgb": "observation.images.workspace",
    "obs/eye_in_hand_rgb": "observation.images.wrist",
}
action_dtype = "float32"
state_dtype = "float32"
use_videos = False

## 3. Create and save a plan

State selector order defines the concatenated `observation.state`. The two source cameras remain separate target features.

In [4]:
plan = create_plan(
    source_path,
    target_root=target_root,
    repo_id="local/libero-close-drawer-demo",
    fps=fps,
    task_metadata="instruction",
    action_source=action_source,
    action_dtype=action_dtype,
    state_sources=state_sources,
    state_dtype=state_dtype,
    image_sources=image_sources,
    use_videos=use_videos,
    adapter="libero",
    selection=episode_selection,
)
pprint(plan.to_dict())
output_root.mkdir(parents=True, exist_ok=True)
if plan_path.exists():
    raise FileExistsError(f"Plan already exists: {plan_path}")
save_plan(plan, plan_path)

{'adapter': 'libero',
 'features': {'action': {'dtype': 'float32', 'shape': [7]},
              'observation.images.workspace': {'dtype': 'image',
                                               'shape': [128, 128, 3]},
              'observation.images.wrist': {'dtype': 'image',
                                           'shape': [128, 128, 3]},
              'observation.state': {'dtype': 'float32', 'shape': [8]}},
 'fps': 20,
 'mappings': {'action': {'cast': 'float32',
                         'operation': 'direct',
                         'sources': ['actions']},
              'observation.images.workspace': {'operation': 'direct',
                                               'sources': ['obs/agentview_rgb']},
              'observation.images.wrist': {'operation': 'direct',
                                           'sources': ['obs/eye_in_hand_rgb']},
              'observation.state': {'cast': 'float32',
                                    'operation': 'concat',
              

PosixPath('/Users/xzhu/Documents/VLA/leport/outputs/libero-close-drawer-demo.yaml')

## 4. Convert

Conversion preflights every mapped field, streams raw source frames, and commits only after LeRobot reload validation succeeds.

In [5]:
if target_root.exists():
    raise FileExistsError(f"Target already exists: {target_root}")
conversion_result = convert(plan_path)
pprint(conversion_result.validation.to_dict())

{'decoded_visual_features': ['observation.images.workspace',
                             'observation.images.wrist'],
 'episode_lengths': [65],
 'features': {'action': {'dtype': 'float32', 'shape': (7,)},
              'episode_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'frame_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'observation.images.workspace': {'dtype': 'image',
                                               'shape': (128, 128, 3)},
              'observation.images.wrist': {'dtype': 'image',
                                           'shape': (128, 128, 3)},
              'observation.state': {'dtype': 'float32', 'shape': (8,)},
              'task_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'timestamp': {'dtype': 'float32', 'names': None, 'shape': (1,)}},
 'repo_id': 'local/libero-close-drawer-demo',
 'root': '/Users/xz

## 5. Validate against the source

The reviewed plan supplies expected episode length, task text, feature schemas, and decoded visual features.

In [6]:
validation_report = validate(target_root, plan=plan_path)
pprint(validation_report.to_dict())

{'decoded_visual_features': ['observation.images.workspace',
                             'observation.images.wrist'],
 'episode_lengths': [65],
 'features': {'action': {'dtype': 'float32', 'shape': (7,)},
              'episode_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'frame_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'observation.images.workspace': {'dtype': 'image',
                                               'shape': (128, 128, 3)},
              'observation.images.wrist': {'dtype': 'image',
                                           'shape': (128, 128, 3)},
              'observation.state': {'dtype': 'float32', 'shape': (8,)},
              'task_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
              'timestamp': {'dtype': 'float32', 'names': None, 'shape': (1,)}},
 'repo_id': 'local/libero-close-drawer-demo',
 'root': '/Users/xz

## 6. Merge converted demonstrations (optional)

Set `RUN_MERGE = True` to convert disjoint `demo_1` with the same schema and merge the two completed LeRobot datasets. Raw HDF5 files are never merge inputs.

In [7]:
# Keep extra conversion and merge work disabled in the default run.
RUN_MERGE = False

merge_result = None
if RUN_MERGE:
    second_plan_path = output_root / "libero-close-drawer-demo-b.yaml"
    second_target = output_root / "libero-close-drawer-demo-b"
    merged_target = output_root / "libero-close-drawer-demo-merged"
    second_plan = create_plan(
        source_path,
        target_root=second_target,
        repo_id="local/libero-close-drawer-demo-b",
        fps=fps,
        task_metadata="instruction",
        action_source=action_source,
        action_dtype=action_dtype,
        state_sources=state_sources,
        state_dtype=state_dtype,
        image_sources=image_sources,
        use_videos=use_videos,
        adapter="libero",
        selection=EpisodeSelection(episode_ids=(f"{task_name}/demo_1",)),
    )
    if second_plan_path.exists() or second_target.exists():
        raise FileExistsError("Second plan or target already exists")
    save_plan(second_plan, second_plan_path)
    second_result = convert(second_plan_path)
    pprint(second_result.validation.to_dict())

    if merged_target.exists():
        raise FileExistsError(f"Merge target already exists: {merged_target}")
    merge_result = merge(
        (target_root, second_target),
        target_root=merged_target,
        repo_id="local/libero-close-drawer-demo-merged",
    )
    pprint(merge_result.to_dict())
else:
    print("Optional merge skipped.")

Optional merge skipped.


## Equivalent CLI commands

Run these equivalent commands from the repository root:

```bash
uv run leport inspect data/libero/libero_90/KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet_demo.hdf5 --adapter libero --episode KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet/demo_0 --json

uv run leport plan \
  --source data/libero/libero_90/KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet_demo.hdf5 \
  --output outputs/libero-close-drawer-demo.yaml \
  --adapter libero --episode KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet/demo_0 \
  --target outputs/libero-close-drawer-demo --repo-id local/libero-close-drawer-demo \
  --fps 20 --task-metadata instruction \
  --action actions --action-dtype float32 \
  --state obs/ee_states --state obs/gripper_states --state-dtype float32 \
  --image obs/agentview_rgb=observation.images.workspace \
  --image obs/eye_in_hand_rgb=observation.images.wrist --no-videos

uv run leport convert --config outputs/libero-close-drawer-demo.yaml --json
uv run leport validate outputs/libero-close-drawer-demo --config outputs/libero-close-drawer-demo.yaml --json

# Optional: plan and convert disjoint demo_1 before merging.
uv run leport plan \
  --source data/libero/libero_90/KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet_demo.hdf5 \
  --output outputs/libero-close-drawer-demo-b.yaml \
  --adapter libero --episode KITCHEN_SCENE5_close_the_top_drawer_of_the_cabinet/demo_1 \
  --target outputs/libero-close-drawer-demo-b --repo-id local/libero-close-drawer-demo-b \
  --fps 20 --task-metadata instruction \
  --action actions --action-dtype float32 \
  --state obs/ee_states --state obs/gripper_states --state-dtype float32 \
  --image obs/agentview_rgb=observation.images.workspace \
  --image obs/eye_in_hand_rgb=observation.images.wrist --no-videos

uv run leport convert --config outputs/libero-close-drawer-demo-b.yaml --json
uv run leport merge outputs/libero-close-drawer-demo outputs/libero-close-drawer-demo-b \
  --target outputs/libero-close-drawer-demo-merged \
  --repo-id local/libero-close-drawer-demo-merged --json
```

> Review existing plan and target paths before rerunning. LePort never overwrites completed outputs silently.